# Analysing EEG Data with MNE-Python

**Author**: Angela Renton, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/air2310"><img src="https://img.shields.io/badge/-Angela_Renton-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

This tutorial demonstrates how to use MNE-Python to load, pre-process, and analyse EEG data. You will work through a complete pipeline in a Jupyter notebook accessed through VSCode within the Neurodesk container, including data loading, preprocessing, event extraction, epoching, ERP computation, and frequency analysis of steady-state visual evoked potentials (SSVEPs).

:::{admonition} Learning Objectives
:class: tip
- Open the MNE-Python environment through the VSCode container in Neurodesk
- Load and pre-process EEG data with MNE
- Epoch data and compute ERPs
- Perform frequency analysis (FFT) to investigate steady-state visual evoked potentials
:::

## Citation and Resources

**MNE-Python:**
: Gramfort, A., et al. (2013). MEG and EEG data analysis with MNE-Python. *Frontiers in Neuroscience*, 7, 267. https://doi.org/10.3389/fnins.2013.00267

**Educational Resources:**
: [MNE documentation](https://mne.tools/stable/auto_tutorials/index.html) | [OSF dataset C689U](https://osf.io/c689u/)

**Dataset:**
: OSF project C689U — 5-channel EEG data from a frequency-tagged attention task with 6 Hz and 7.5 Hz visual stimuli

## Prerequisites

:::{warning}
Before starting this tutorial, ensure you have:
:::

- [ ] Running Neurodesk container
- [ ] Familiarity with Python programming
- [ ] Basic understanding of EEG concepts (channels, sampling rate, frequency)

## Getting Started

To open MNE-Python in VSCode, navigate to the Neurodesk application menu:

**Neurodesk → Electrophysiology → mne → vscodeGUI 0.23.4**

**Important:** You must use this specific version of VSCode to access the MNE conda environment.

![Opening the MNE-Python VSCode container from the application menu](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut0.png)
*Opening the MNE-Python container (vscodeGUI) from the Neurodesk application menu.*

VSCode will launch inside the Neurodesk container with the MNE conda environment available:

![VSCode launched with the MNE conda environment available](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut1.png)
*VSCode launched inside the Neurodesk container with the MNE conda environment available.*

## Creating a Jupyter Notebook

Open the folder `/home/user/Desktop/storage` (or a subfolder of your choice) in VSCode, then create a new file:

![Creating a new Jupyter notebook file in VSCode](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut2.png)
*Creating a new Jupyter notebook file in VSCode.*

If this is your first time using Jupyter in VSCode, install the required extensions when prompted:

![VSCode prompt to install Jupyter extensions](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut3.png)
*VSCode prompt to install the Jupyter notebook extensions.*

## Selecting the MNE Python Kernel

Click the "Select Kernel" button in the top right corner of the notebook:

![Selecting the Python kernel in VSCode](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut4.png)
*Selecting the Python kernel in the top-right corner of the VSCode notebook.*

From the dropdown menu, select `mne-0.23.4`:

![Selecting the MNE-Python kernel from the dropdown](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut5.png)
*Selecting the mne-0.23.4 kernel from the dropdown menu.*

## Activating the MNE Conda Environment

Open a terminal in VSCode (Terminal → New Terminal or Ctrl+Shift+`).

If this is your first time, initialize conda:

```bash
conda init bash
```

Close and reopen the terminal, then activate the MNE environment:

```bash
conda activate mne-0.23.4
```

## Downloading Sample Data

In the terminal, install the OSF client and download the sample dataset:

```bash
pip install osfclient
osf -p C689U fetch Data_sample.zip ~/neurodesktop-storage/EEGDEMO/Data_sample.zip
unzip Data_sample.zip
```

**Note:** Update the path to match your storage location. The dataset contains 5 EEG channels from one participant viewing a frequency-tagged visual display with 6 Hz and 7.5 Hz stimuli.

## Loading and Visualising Raw Data

In your Jupyter notebook, start by setting up the interactive plotting backend and importing necessary libraries:

```python
%matplotlib qt
import os
import numpy as np
import mne

sample_data_folder = '~/neurodesktop-storage/EEGDemo/Data_sample'
sample_data_raw_file = os.path.join(sample_data_folder, 'sub-01', 'eeg',
                                    'sub-01_task-FeatAttnDec_eeg.vhdr')
raw = mne.io.read_raw_brainvision(sample_data_raw_file, preload=True)
print(raw)
print(raw.info)
```

Visualise the raw EEG data:

```python
raw.plot()
```

![Interactive EEG data viewer in MNE](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut6.png)
*Interactive EEG data viewer showing the raw time series.*

## Setting Up Electrode Montage

Define the standard electrode positions for the 5-channel EEG setup:

```python
montage = {'Iz': [0, -110, -40], 'Oz': [0, -105, -15], 'POz': [0, -100, 15],
           'O1': [-40, -106, -15], 'O2': [40, -106, -15]}
montageuse = mne.channels.make_dig_montage(ch_pos=montage,
    lpa=[-82.5, -19.2, -46], nasion=[0, 83.2, -38.3], rpa=[82.2, -19.2, -46])
```

## Extracting Events

Extract trigger events from the TRIG channel:

```python
trigchan = raw.copy().pick('TRIG')
trigchan._data = trigchan._data * 1000000
events = mne.find_events(trigchan, stim_channel='TRIG', consecutive=True,
                         initial_event=True, verbose=True)
mne.viz.plot_events(events, raw.info['sfreq'], raw.first_samp)
```

![Event markers extracted from the EEG data](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut7.png)
*Event markers extracted from the TRIG channel.*

## Pre-processing EEG Data

Pre-process the EEG channels by applying the electrode montage, interpolating bad channels, and filtering:

```python
eeg_data = raw.copy().pick_types(eeg=True, exclude=['TRIG'])
eeg_data.info.set_montage(montageuse)
eeg_data_interp = eeg_data.copy().interpolate_bads(reset_bads=True)
eeg_data_interp.filter(l_freq=1, h_freq=45, h_trans_bandwidth=0.1)
eeg_data_interp.plot(events=events, duration=10.0, scalings=dict(eeg=0.00005),
                     color='k', event_color='r')
```

![Pre-processed EEG data with event markers](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut8.png)
*Pre-processed EEG data with event markers overlaid.*

## Epoching and ERP Computation

Segment the continuous data into epochs aligned to trial events and compute event-related potentials (ERPs):

```python
event_id = {'attend 6Hz K': 23, 'attend 7.5Hz K': 27}
epochs = mne.Epochs(eeg_data_interp, events, event_id=event_id, tmin=0,
                    tmax=15, baseline=(0, 0), reject=dict(eeg=0.000400), detrend=1)
epochs.drop_bad()

attend6 = epochs['attend 6Hz K'].average()
attend75 = epochs['attend 7.5Hz K'].average()
evokeds = dict(attend6=list(epochs['attend 6Hz K'].iter_evoked()),
               attend75=list(epochs['attend 7.5Hz K'].iter_evoked()))
mne.viz.plot_compare_evokeds(evokeds, combine='mean')
```

![ERP comparison between attend-6Hz and attend-7.5Hz conditions](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut9.png)
*ERP comparison showing frequency-tagged steady-state responses.*

## Frequency Analysis (FFT)

Perform Fast Fourier Transform (FFT) on the averaged ERP data to examine the frequency spectrum and identify steady-state visual evoked potentials (SSVEPs):

```python
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt

n_samples = attend6.data.shape[1]
sampling_freq = 1200
epochs_np = np.empty((n_samples, 2))
epochs_np[:, 0] = attend6.data.mean(axis=0)
epochs_np[:, 1] = attend75.data.mean(axis=0)

fftdat = np.abs(fft(epochs_np, axis=0)) / n_samples
freq = fftfreq(n_samples, d=1 / sampling_freq)

fig, ax = plt.subplots(1, 1)
ax.plot(freq, fftdat[:, 0], '-', label='attend 6Hz', color=[78/255, 185/255, 159/255])
ax.plot(freq, fftdat[:, 1], '-', label='attend 7.5Hz', color=[236/255, 85/255, 58/255])
ax.set_xlim(4, 17)
ax.set_ylim(0, 1e-6)
ax.set_title('Frequency Spectrum')
ax.legend()
```

![FFT frequency spectrum showing SSVEP modulation by attention](/static/tutorials/electrophysiology/eeg_mne-python/EEGtut10.png)
*Frequency spectrum showing that the SSVEP amplitude is larger at the attended frequency.*

## Summary

1. Opened MNE-Python through VSCode in the Neurodesk container
2. Created and configured a Jupyter notebook with the mne-0.23.4 kernel
3. Downloaded EEG sample data from the OSF repository
4. Loaded raw BrainVision EEG data and inspected the signal
5. Set electrode montage for spatial information
6. Extracted trigger events from the stimulus channel
7. Pre-processed EEG data (filtering, interpolation, montage application)
8. Epoched data and computed event-related potentials for two experimental conditions
9. Performed frequency analysis (FFT) to examine steady-state visual evoked potentials and demonstrate attention modulation

:::{{seealso}}
For an alternative approach to analysing electrophysiological data, see [Analysing M/EEG Data with FieldTrip](fieldtrip.ipynb)
:::